In [1]:
! pip install duckdb

In [3]:
import duckdb

try:
    con.close() 
except NameError:
    pass

con = duckdb.connect("../dataset/goodreads.db") 
con.sql("SET memory_limit='6GB'")
con.sql("SET temp_directory='duckdb_tmp'")

In [5]:
con.sql("""
    CREATE OR REPLACE TABLE books_raw AS
    SELECT
        book_id,
        title,
        description,
        language_code,
        publisher,
        format,
        is_ebook,
        country_code,
        average_rating,
        ratings_count,
        text_reviews_count,
        num_pages,
        publication_year,
        len(series) AS n_series,
        len(authors) AS n_authors,
        authors[1].author_id AS first_author_id
    FROM read_json_auto(
        '../dataset_original/goodreads_books.json',
        format='newline_delimited',
        maximum_object_size=20000000)
""")

con.sql("SELECT COUNT(*) AS n_books FROM books_raw").show()

┌─────────┐
│ n_books │
│  int64  │
├─────────┤
│ 2360655 │
└─────────┘



In [6]:
con.sql("SHOW TABLES").show()

┌───────────┐
│   name    │
│  varchar  │
├───────────┤
│ books_raw │
└───────────┘



In [7]:
con.sql("""
    SELECT
        COUNT(*) AS n_rows,
        SUM(CASE WHEN title IS NULL OR title = '' THEN 1 ELSE 0 END) AS title_missing,
        SUM(CASE WHEN description IS NULL OR description = '' THEN 1 ELSE 0 END) AS description_missing,
        SUM(CASE WHEN language_code IS NULL OR language_code = '' THEN 1 ELSE 0 END) AS language_missing,
        SUM(CASE WHEN publisher IS NULL OR publisher = '' THEN 1 ELSE 0 END) AS publisher_missing,
        SUM(CASE WHEN format IS NULL OR format = '' THEN 1 ELSE 0 END) AS format_missing,
        SUM(CASE WHEN num_pages IS NULL OR num_pages = '' THEN 1 ELSE 0 END) AS pages_missing,
        SUM(CASE WHEN publication_year IS NULL OR publication_year = '' THEN 1 ELSE 0 END) AS year_missing,
        SUM(CASE WHEN average_rating IS NULL OR average_rating = '' THEN 1 ELSE 0 END) AS avg_rating_missing,
        SUM(CASE WHEN ratings_count IS NULL OR ratings_count = '' THEN 1 ELSE 0 END) AS ratings_count_missing,
        SUM(CASE WHEN first_author_id IS NULL OR first_author_id = '' THEN 1 ELSE 0 END) AS author_missing
    FROM books_raw
""").show()

┌─────────┬───────────────┬─────────────────────┬──────────────────┬───────────────────┬────────────────┬───────────────┬──────────────┬────────────────────┬───────────────────────┬────────────────┐
│ n_rows  │ title_missing │ description_missing │ language_missing │ publisher_missing │ format_missing │ pages_missing │ year_missing │ avg_rating_missing │ ratings_count_missing │ author_missing │
│  int64  │    int128     │       int128        │      int128      │      int128       │     int128     │    int128     │    int128    │       int128       │        int128         │     int128     │
├─────────┼───────────────┼─────────────────────┼──────────────────┼───────────────────┼────────────────┼───────────────┼──────────────┼────────────────────┼───────────────────────┼────────────────┤
│ 2360655 │             7 │              412233 │          1060153 │            654362 │         646754 │        764133 │       599625 │                524 │                   524 │            537 │
└────

In [8]:
#numeric columns are actually stored as text and need to be converted to int
con.sql("""
    SELECT
        MIN(TRY_CAST(num_pages AS INTEGER)) AS min_pages,
        MAX(TRY_CAST(num_pages AS INTEGER)) AS max_pages,
        MEDIAN(TRY_CAST(num_pages AS INTEGER)) AS median_pages,
        MIN(TRY_CAST(publication_year AS INTEGER)) AS min_year,
        MAX(TRY_CAST(publication_year AS INTEGER)) AS max_year,
        MEDIAN(TRY_CAST(publication_year AS INTEGER)) AS median_year,
        MIN(TRY_CAST(average_rating AS DOUBLE)) AS min_avg,
        MAX(TRY_CAST(average_rating AS DOUBLE)) AS max_avg,
        MAX(TRY_CAST(ratings_count AS INTEGER)) AS max_ratings_count
    FROM books_raw
""").show()

┌───────────┬───────────┬──────────────┬──────────┬──────────┬─────────────┬─────────┬─────────┬───────────────────┐
│ min_pages │ max_pages │ median_pages │ min_year │ max_year │ median_year │ min_avg │ max_avg │ max_ratings_count │
│   int32   │   int32   │    double    │  int32   │  int32   │   double    │ double  │ double  │       int32       │
├───────────┼───────────┼──────────────┼──────────┼──────────┼─────────────┼─────────┼─────────┼───────────────────┤
│         0 │    945077 │        245.0 │        0 │    65535 │      2011.0 │     0.0 │     5.0 │           4899965 │
└───────────┴───────────┴──────────────┴──────────┴──────────┴─────────────┴─────────┴─────────┴───────────────────┘



In [9]:
#build a new clean books table

#convert text columns to numeric values using cast
#turn into empty strings to null or unkown
#turn junk values (like out of range page coutns and years) into null
#drop the 7 books with no title
con.sql("SET memory_limit='4GB'")
con.sql("""
    CREATE OR REPLACE TABLE books_clean AS
    SELECT
        CAST(book_id AS BIGINT) AS book_id,
        title,
        NULLIF(description, '') AS description,
        COALESCE(NULLIF(language_code, ''), 'unknown') AS language_code,
        NULLIF(publisher, '') AS publisher,
        NULLIF(format, '') AS format,
        (is_ebook = 'true') AS is_ebook,
        NULLIF(country_code, '') AS country_code,
        TRY_CAST(average_rating AS DOUBLE) AS avg_rating,
        TRY_CAST(ratings_count AS BIGINT) AS ratings_count,
        TRY_CAST(text_reviews_count AS BIGINT) AS text_reviews_count,
        CASE WHEN TRY_CAST(num_pages AS INTEGER) BETWEEN 10 AND 5000
             THEN TRY_CAST(num_pages AS INTEGER) END AS num_pages,
        CASE WHEN TRY_CAST(publication_year AS INTEGER) BETWEEN 1400 AND 2026
             THEN TRY_CAST(publication_year AS INTEGER) END AS publication_year,
        (n_series > 0) AS in_series,
        n_authors,
        CAST(NULLIF(first_author_id, '') AS BIGINT) AS first_author_id
    FROM books_raw
    WHERE title IS NOT NULL AND title <> ''
""")

con.sql("SELECT COUNT(*) AS n_books_clean FROM books_clean").show()

con.sql("""
    SELECT
        SUM(CASE WHEN num_pages IS NULL THEN 1 ELSE 0 END) AS pages_null,
        SUM(CASE WHEN publication_year IS NULL THEN 1 ELSE 0 END) AS year_null,
        MIN(num_pages) AS min_pages, MAX(num_pages) AS max_pages,
        MIN(publication_year) AS min_year, MAX(publication_year) AS max_year
    FROM books_clean
""").show()

┌───────────────┐
│ n_books_clean │
│     int64     │
├───────────────┤
│       2360648 │
└───────────────┘

┌────────────┬───────────┬───────────┬───────────┬──────────┬──────────┐
│ pages_null │ year_null │ min_pages │ max_pages │ min_year │ max_year │
│   int128   │  int128   │   int32   │   int32   │  int32   │  int32   │
├────────────┼───────────┼───────────┼───────────┼──────────┼──────────┤
│     785561 │    602066 │        10 │      5000 │     1400 │     2025 │
└────────────┴───────────┴───────────┴───────────┴──────────┴──────────┘

